In [ ]:
# --- path configuration -------------------------------------------------
# Every path goes through resolve() (inputs under data/) or out_path() (this run's output).
# Set SURFACEOME_DATA to point at the data archive; see README.md.
from surfaceome_config import resolve, out_path


## Merging and Cleaning of CellphoneDB human cell-cell interaction database

The goal of this notebook is to merge the datafrom the cellphoneDB (https://www.cellphonedb.org/, a cell-cell interaction database), filtering for trans interactions, and merging with the human protein size database in order to predict gap sizes between cells

In [ ]:
import pandas as pd
import numpy as np
import re as re

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
#import cellphoneDB datafiles

#protein complex components
complx = pd.read_csv(str(resolve('inputs/cellphonedb/complex_curated.csv')))
#with the complexes curated here (the MHC class II heterodimers), each with its citation
from surfaceomeTopography.interactions import with_curated_complexes
complx = with_curated_complexes(complx, pd.read_csv(resolve('curated/curated_complexes.csv')))

#protein-protein interactions
intact = pd.read_csv(str(resolve('inputs/cellphonedb/interaction_curated.csv')))

#proteins in database and protein localization annotations
prots = pd.read_csv(str(resolve('inputs/cellphonedb/protein_curated.csv')))

#hla to uniprot mapping
hla = pd.read_csv(str(resolve('inputs/cellphonedb/hla_curated.csv')))

## Filtering for trans interactions

Given the database organization what is the best way to merge with the human protein heights?

could start with the complex dataframe given its mapping to uniprot ids already. Then assign a height to the complex based on the maximum height within the complex. This could then be merged with the interaction dataframe.

##For filtering for interactions that are in trans only, take a simular strategy mergeing on uniprot for the complexes etc and having an annotation that can then be used to filter out interactions that don't have transmembrane domains on both sides###

Actually don't need, secreted complexes are already annotated!!
Just filter out secreted == 'True'

In [ ]:
#select transmembrane complexes, no secreted or peripheral complexes identified
tm_complx = complx[(complx['transmembrane'] == True)]

tm_complx.describe()

In [ ]:
#select TM proteins
tm_prots = prots[(prots['transmembrane'] == True)]

tm_prots.describe()

Next after filtering for transmembrane protiens merge size information into the complexes and the individual proteins

human surfaceome protein sizes: UP000005640

In [ ]:
#Per-protein heights to size the CellphoneDB interactions from, built from THIS run's height
#table. This notebook used to read them from a file Human Interaction Heights writes -- but that
#notebook runs after this one, so every run sized its interactions from the stored 2022 copy.
from surfaceomeTopography.interactions import protein_heights_for_interactions
names = pd.concat([pd.read_csv(resolve('inputs/mapping/uniprot_surfaceome_gene_name.tab'), sep = '\t'),
                   pd.read_csv(resolve('inputs/mapping/uniprot_mapping_cleanup.tab'), index_col=None, sep = '\t')])
sizes = protein_heights_for_interactions(
    pd.read_csv(str(resolve('database/height_estimates.csv'))),
    pd.read_csv(resolve('inputs/proteome/surfaceome_current.tsv'), sep = '\t', header = None, names = ['current', 'af_id']),
    names)
sizes.to_csv(out_path('database/protein_heights_for_interactions.csv'))


In [ ]:
#only columns I want
sizes_well = sizes[['ID link','Entry name', 'af_id', 'methods_<lambda>','total_height']]

sizes_well.tail()

In [ ]:
tm_complx_cleaned = tm_complx[['complex_name','uniprot_1','uniprot_2','uniprot_3','uniprot_4','transmembrane','peripheral','secreted','comments_complex']]
tm_complx_cleaned.head()

In [ ]:
tm_complx_sizes = pd.merge(tm_complx_cleaned, sizes_well, left_on ='uniprot_1', right_on ='ID link', 
                           how = 'left', suffixes = ('', '_1'))

tm_complx_sizes = pd.merge(tm_complx_sizes, sizes_well, left_on ='uniprot_2', right_on ='ID link', 
                           how = 'left', suffixes = ('_1', '_2'))

tm_complx_sizes = pd.merge(tm_complx_sizes, sizes_well, left_on ='uniprot_3', right_on ='ID link', 
                           how = 'left', suffixes = ('_2', '_3'))

tm_complx_sizes = pd.merge(tm_complx_sizes, sizes_well, left_on ='uniprot_4', right_on ='ID link', 
                           how = 'left', suffixes = ('_3', '_4'))

In [ ]:
#export tm complexes with sizes
#explore csv file
tm_complx_sizes.to_csv(out_path('database/cellphonedb_tm_complexes_sizes.csv'))

In [ ]:
#check for max size in complex, assign as complex height
tm_complx_sizes['complex_height'] = tm_complx_sizes[["total_height_1", "total_height_2", "total_height_3", 'total_height_4']].max(axis=1)
tm_complx_sizes['complex_max'] = tm_complx_sizes[["total_height_1", "total_height_2", "total_height_3", 'total_height_4']].idxmax(axis=1)
complx_sizes = tm_complx_sizes[~tm_complx_sizes['complex_height'].isna()].copy().reset_index(drop=True)

complx_sizes.loc[:,'uniprot'] = 'uniprot_'+complx_sizes['complex_max'].str[-1:].copy()
complx_sizes.loc[:,'Entry'] = 'Entry name_'+complx_sizes['complex_max'].str[-1:].copy()


#retrieve and assign Entry and ID link as the tallest protein within the complex
l = []
e = []

for row in complx_sizes.itertuples():
    l.append(complx_sizes.loc[row.Index, row.uniprot])
    e.append(complx_sizes.loc[row.Index, row.Entry])
    
ID_link = pd.Series(l)
Entry = pd.Series(e)

complx_sizes.loc[:,'ID link'] = ID_link
complx_sizes.loc[:,'Entry'] = Entry

complx_sizes.head()

In [ ]:
tm_prots_cleaned = tm_prots[['uniprot', 'protein_name']]

tm_prots_sizes = pd.merge(tm_prots_cleaned, sizes_well, left_on ='uniprot', right_on ='ID link', 
                           how = 'left')
tm_prots_sizes.tail()

In [ ]:
#now complex sizes merged and proteins sizes merged, now merge with interactions
intact_sizes = pd.merge(intact, complx_sizes[['complex_name', 'complex_height', 'ID link', 'Entry']], left_on =['partner_a'], right_on =['complex_name'], 
                           how = 'left', suffixes = ('', '_a'))

intact_sizes = pd.merge(intact_sizes, complx_sizes[['complex_name', 'complex_height', 'ID link','Entry']], left_on ='partner_b', right_on ='complex_name', 
                           how = 'left', suffixes = ('_a', '_b'))

intact_sizes = pd.merge(intact_sizes, tm_prots_sizes[['uniprot', 'total_height']], left_on ='partner_a', right_on ='uniprot', 
                           how = 'left', suffixes = ('_', '_a'))

intact_sizes = pd.merge(intact_sizes, tm_prots_sizes[['uniprot', 'total_height']], left_on ='partner_b', right_on ='uniprot', 
                           how = 'left', suffixes = ('_a', '_b'))
    
intact_sizes.tail()

In [ ]:
intact_sizes.to_csv(out_path('database/cellphonedb_interaction_sizes.csv'))

In [ ]:
intact_sizes['intact_height'] = intact_sizes[["complex_height_a", "complex_height_b", "total_height_a", 'total_height_b']].sum(axis=1)

intact_sizes['intact_height_count'] = intact_sizes[["complex_height_a", "complex_height_b", "total_height_a", 'total_height_b']].count(axis=1)

measured_int = intact_sizes[intact_sizes['intact_height_count'] > 1]

measured_int

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure();

measured_int['intact_height'].plot.hist(alpha=0.5
                           ,bins=range(0, 50, 1) 
                           #,density = True
                          )

In [ ]:
#get id link for each interaction
measured_int = measured_int.assign(ID_link_prot1 = np.where(measured_int['ID link_a'].notnull(), measured_int['ID link_a'], measured_int['uniprot_a']))
measured_int = measured_int.assign(ID_link_prot2 = np.where(measured_int['ID link_b'].notnull(), measured_int['ID link_b'], measured_int['uniprot_b']))
measured_int.loc[:,'Human interaction_id'] = measured_int.apply(lambda row: ','.join(sorted([row['ID_link_prot1'], row['ID_link_prot2']])), axis = 1)

In [ ]:
#get gene name for each interaction
measured_int = measured_int.assign(Entry_prot1 = np.where(measured_int['Entry_a'].notnull(), measured_int['Entry_a'], measured_int['protein_name_a']))
measured_int = measured_int.assign(Entry_prot2 = np.where(measured_int['Entry_b'].notnull(), measured_int['Entry_b'], measured_int['protein_name_b']))

In [ ]:
#assign best height for each interaction
measured_int = measured_int.assign(total_height_a = np.where(measured_int['total_height_a'].notnull(), measured_int['total_height_a'], measured_int['complex_height_a']))
measured_int = measured_int.assign(total_height_b= np.where(measured_int['total_height_b'].notnull(), measured_int['total_height_b'], measured_int['complex_height_b']))

In [ ]:
measured_int.to_csv(out_path('database/cellphonedb_tm_interaction_sizes.csv'), index = False)